**Date:** 11/23/2024

## NLP Clustering — KMeans, DBSCAN & TSNE

**Instructions**

**0.1 BRIEF LECTURE**: About Cluster Analysis

**0.2 IMPORT PACKAGES** 
- pandas
- CountVectorizer from sklearn.feature_extraction.text) # conda install -c conda-forge scikit-learn
- TSNE from sklearn.manifold 
- DBSCAN  from sklearn.cluster
- KMeans from sklearn.cluster 
- plotly.graph_objects

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer # Document Term Basis 
from sklearn.manifold import TSNE # Dimensionality Reducer
from sklearn.cluster import KMeans # Clustering
from sklearn.cluster import DBSCAN # Clustering 
import plotly.graph_objects as go 





**Problem 0.3. READ DATAFRAME** 
- Print the column names and the shape.

In [ ]:
folder = 'ai-coding-discussion.csv'
filename = 'ai-coding-discussion.csv'
fullpath = f'{folder}/{filename}'

df = pd.read_csv(fullpath)
print(df.columns)
print(df.shape)


**Problem 0.4. CLEAN DATA**
- Clean the `date` column 
- Clean the `parent_id` column, converting empty cells to 'TOPIC'
- Clean the `text` column, by converting empty cells to 'BLANK' 
- Display first 3 rows of `df`

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['parent_id'] = df['parent_id'].fillna('TOPIC')
df['text'] = df['text'].fillna('BLANK')

df.head(3)



**Problem 0.5. ADDITIONAL DATA WRANGLING**

- Pull out the text column into the variable `text`
- Display the first 5 rows

In [ ]:
text = df['text']
text.head(5)

**Problem 1. DATA TRANSFORMATION**: Document Term Matrix 
- Create a document-term matrix based on `text` named `dtm`
- Extract the matrix into `mat`
- Extract the column names (feature names) into `col`
- Create a dataframe of the document-term matrix named `df_dtm`
- Display the first five rows in `df_dtm`

In [ ]:
vectorizer = CountVectorizer()
dtm = vectorizer.fit_transform(text)

mat = dtm.toarray()
col = vectorizer.get_feature_names_out()

df_dtm = pd.DataFrame( mat, columns= col)
df_dtm.index = text
df_dtm.head(5)

**PROBLEM 2. DIMENSIONALITY REDUCTION**: TSNE
- Instanstiate a TSNE object named `tsne` with 2 dimensions (components), a random state of 42, and a perplexity of 5
- Reduce the dimensionality and store it in `xy`
- Create a dataframe, `df2d`, based on `xy` with columns `x`, and `y` 
- Add `text` to `df2d` 
- Display `df2d`

In [ ]:
tsne =TSNE(n_components= 2, random_state= 42, perplexity= 5)
xy = tsne.fit_transform(mat)
df2d = pd.DataFrame(xy, columns=['x','y'])
df2d['text'] = text 
df2d

**PROBLEM 3. DATA VISUALIZATION:** TSNE 
- Visualize your dataset as a scatter plot so that when you hover over a point, you can see the post associated with the point.

In [ ]:
big_text = []
for i , post in enumerate(text) :
    new_post = [ ]
    br_point = 10
    words = post.split()
    for j,word in enumerate(words): 
        new_post.append(word)
        if (j+1) % br_point ==0 :
            new_post.append('<br>')
            new_text = ' '.join(new_post)
            big_text.append(new_text)
          
fig = go.Figure()

fig.add_trace(
    go.Scatter (x = df2d['x'], y = df2d['y'],
               mode = 'markers' , text = big_text)

)
fig.update_layout(
                title = {'text' : 'Data for Ai coding', 'x' :.5},
                  xaxis_title = 'Positivity vs Negativity',
                  yaxis_title = 'True vs False',
                  )
                 
fig.show()

**PROBLEM 4. CLUSTERING & VISUALIZATION:** KMEANS 
- Instantiate a `kmeans` object with 50 clusters
- Cluster your document term matrix, storing the groups in `groups`
- Add groups to `df2d['group']`
- Visualize the clusters using the `Rainbow` colorscale

In [ ]:
num_clusters = 50 
kmeans = KMeans(n_clusters= num_clusters , random_state=42)
groups = kmeans.fit_predict(dtm)
df2d['group'] = groups

fig = go.Figure()
fig.add_trace(go.Scatter(
              x = df2d['x'],
              y = df2d['y'],
              text = big_text,
              mode = 'markers',
              marker = dict(color = df2d['group'], colorscale = 'rainbow')))
fig.show()


**PROBLEM 5. CLUSTERING & VISUALIZATION:** DBSCAN
- Instantiate a `dbscan` object. Experiment with eps and min_samples to get a good clustering
- Cluster your document term matrix, storing the groups in `groups`
- Add groups to `df2d['group']`
- Visualize the clusters using the `Rainbow` colorscale

In [ ]:
dbscan = DBSCAN(eps=.5 , min_samples= 6, metric =  'cosine')
groups = dbscan.fit_predict(dtm)
groups 
df2d['group'] = groups
df2d


fig = go.Figure()
fig.add_trace(go.Scatter(
              x = df2d['x'],
              y = df2d['y'],
              text = big_text,
              mode = 'markers',
              marker = dict(color = df2d['group'], colorscale = 'rainbow')))
fig.show()

cond = df2d['group']!= -1
dfno = df2d [cond]
dfno


#dbscan_bigtext = big_text[cond]
big_text_no = [txt for txt,cnd in zip(big_text,cond) if  cnd]

fig = go.Figure()
fig.add_trace(go.Scatter(
              x = dfno['x'],
              y = dfno['y'],
              text = big_text_no,
              mode = 'markers',
              marker = dict(color = dfno['group'], colorscale = 'rainbow')))
fig.show()
